# Sentiment Analysis of Financial Text Data from News Article Summaries from Alpha Vantage using the Huggingface transformer library.


### Typical LLM Process

* Usually you have some sort of "base" pre-trained model and then you fine-tune it to your specific dataset.

##### Pre-training: (MASSIVE TRAINING SET!)
* Involves training a model on a large, generic dataset to learn broad language understanding, without focusing on specific tasks.
* For ChatGPT you train on good chunk of the internet, you get an Internet document completer with undefined behavior.

#### Fine-tuning: (SMALL, SPECIFIC TRAINING SET!)
* Adapts the pre-trained model to a specific task by training on a smaller, task-specific dataset, optimizing the model for particultar objectives.
* With ChatGPT fine tune on assistant training data where question is on top and answer is below. A question-answerer in other words.
* For example, in our case financial news sentiment classfication of "Bearish","Bullish", or "Neutral" would be the task.

#### Reinforcement Learning From Human FeedBack
* Let model respond and different raters rate which response is better so can predict which candidate response is more desirable and run PPO to fine tune this sampling policy so answers that ChatGPT gives better answer with respect to reward.

### Hugging Face
"
👋 Hi!

We are on a mission to democratize good machine learning, one commit at a time.
"

https://huggingface.co/

### Hugging Face Transformer Library
* Comprehensive open-source AI library across various domains like Natural Language Processing (NLP) + Computer Vision + Audio.
  - It has many pre-trained an fine tuned models in its model hub. https://huggingface.co/models
  - It has tools for fine-tuning and building upon pre-trained models
* User-friendly: Eases the creation and deployment of NLP models with PyTorch and TensorFlow integration for customization.
* Versatile: Includes multilingual support and offers comprehensive tools to build upon existing models.

### BERT (Bidirectional Encoder Representations from Transformers) by Google as base model
* A revolutionary pre-trained language model that captures contextual relationships in text, enabling state-of-the-art performance across various natural language processing tasks
* Bert Pretraining:
  - Masked Language Model (MLM) randomly hides some words in a sentence and trains the model to predict them based on their context
  - Next Sentence Prediction (NSP) involves understanding the relationship between two sentences.

### Finbert
* "Pre-trained BERT knew how to talk, but now it was time to teach it how to talk like a trader. We took the pre-trained BERT and then further trained it on a purely financial corpus called Reuters TRC2."
* Fine-tuning with labeled data for financial sentiment classification with the Financial Phrasebank. "It is a very well thought-out and carefully labeled albeit a small dataset. Researchers extracted 4500 sentences from various news articles, which include financial terms. Then 16 experts and master students with finance backgrounds labeled them. They didn’t only report labels but also inter-annotator agreement level for each sentence, which means how many experts labelled as positive, neutral and negative."
*The later Yi Yang et al released finbert-tone model "is the FinBERT model fine-tuned on 10,000 manually annotated (positive, negative, neutral) sentences from analyst reports. This model achieves superior performance on financial tone analysis task."
* https://github.com/yya518/
* https://huggingface.co/yiyanghkust/finbert-tone

### Goal of this video

#### 1. Do sentiment analysis on alpha-vantage dataset using pre-trained FinBert model using Transformers' *Pipeline* module which basically abstracts everything into:
  * Input: Sentence.
  * Output: Sentiment Score.

#### 2. Fine-tune this model to alpha-vantage dataset using Transformers' *Trainer* module.

## Preliminaries
### Need a GPU
### Installs necessary libraries then restart

In [ ]:
!pip install --upgrade accelerate datasets

     ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 297.4/297.4 kB 4.9 MB/s eta 0:00:00
     ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 510.5/510.5 kB 12.8 MB/s eta 0:00:00
     ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 116.3/116.3 kB 6.3 MB/s eta 0:00:00
     ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 194.1/194.1 kB 4.3 MB/s eta 0:00:00
     ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 134.8/134.8 kB 10.4 MB/s eta 0:00:00
  Using cached nvidia_cuda_nvrtc_cu12-12.1.105-py3-none-manylinux1_x86_64.whl (23.7 MB)
  Using cached nvidia_cuda_runtime_cu12-12.1.105-py3-none-manylinux1_x86_64.whl (823 kB)
  Using cached nvidia_cuda_cupti_cu12-12.1.105-py3-none-manylinux1_x86_64.whl (14.1 MB)
  Using cached nvidia_cudnn_cu12-8.9.2.26-py3-none-manylinux1_x86_64.whl (731.7 MB)
  Using cached nvidia_cublas_cu12-12.1.3.1-py3-none-manylinux1_x86_64.whl (410.6 MB)
  Using cached nvidia_cufft_cu12-11.0.2.54-py3-none-manylinux1_x86_64.whl (121.6 MB)
  Using cached nvidia_curand_cu12-10.3.2.106-py3-none-manylinux1_x86_

#### Imports

In [ ]:
# Pytorch Deep Learning
import torch
# Pandas+Numpy
import numpy as np
import pandas as pd
# Sklearn metrics
from sklearn.metrics import balanced_accuracy_score,accuracy_score

# Hugging Face Transformer Libraries
from transformers import AutoModelForSequenceClassification, AutoTokenizer, pipeline,Trainer, TrainingArguments
# Hugging Face Datasets
from datasets import Dataset

##### Check if GPU is available and set it to device variable

In [ ]:
if torch.cuda.is_available():
    print("CUDA available. GPU will be used for computation.")
    device = 0  # Default to the first GPU; adjust if you have multiple GPUs
else:
    print("CUDA not available. Using CPU for computation.")
    device = -1  # Indicates CPU usage


CUDA available. GPU will be used for computation.


### Load Alpha Vantage Sentiment Dataset
*

In [ ]:
df = pd.read_csv("alpha_vantage_sentiment_TSLA.csv")
df.head()

,title,url,time_published,authors,summary,banner_image,source,category_within_source,source_domain,topics,overall_sentiment_score,overall_sentiment_label,ticker_sentiment,ticker_relevance_TSLA,ticker_sentiment_TSLA,num_tickers,sentiment_ewm,Original_Sentiment_Label_Text,Sentiment_Label_Text,Label
0,"Elon Musk Making Friends With Big Oil, But May...",https://www.thestreet.com/investing/elon-musk-...,2022-03-05 15:20:00,['Luc Olinga'],Tesla's CEO has pushed the automotive sector t...,https://www.thestreet.com/.image/t_share/MTg0O...,The Street,NaN,www.thestreet.com,"[{'topic': 'Economy - Monetary', 'relevance_sc...",-0.420743,Bearish,"[{'ticker': 'TSLA', 'relevance_score': '0.3850...",0.385032,-0.344242,1.0,-0.344242,Somewhat-Bearish,Negative,1
1,Tesla bull says approval of Berlin Gigafactory...,https://www.cnbc.com/2022/03/07/tesla-berlin-g...,2022-03-07 11:50:55,['Sam Shead'],Tesla's market cap soared to over $1 trillion ...,https://image.cnbcfm.com/api/v1/image/10699632...,CNBC,Top News,www.cnbc.com,"[{'topic': 'Earnings', 'relevance_score': '0.1...",-0.155992,Somewhat-Bearish,"[{'ticker': 'TSLA', 'relevance_score': '0.5197...",0.519739,-0.122564,1.0,-0.232849,Neutral,Neutral,2
2,Tesla Bull Dan Ives Sees 70% Upside In Tesla,https://www.benzinga.com/news/22/03/26017135/t...,2022-03-07 14:27:24,[],"Tesla Inc ( NASDAQ: TSLA ) overcame a ""major...",https://cdn.benzinga.com/files/imagecache/og_i...,Benzinga,News,www.benzinga.com,"[{'topic': 'Earnings', 'relevance_score': '0.1...",-0.019120,Neutral,"[{'ticker': 'TSLA', 'relevance_score': '0.9248...",0.924869,0.014649,1.0,-0.149523,Neutral,Neutral,2
3,Tesla Whale Trades For March 07,https://www.benzinga.com/markets/options/22/03...,2022-03-07 18:11:35,['Benzinga Insights'],A whale with a lot of money to spend has taken...,https://www.benzinga.com/files/images/story/20...,Benzinga,Markets,www.benzinga.com,"[{'topic': 'Earnings', 'relevance_score': '0.1...",0.118655,Neutral,"[{'ticker': 'TSLA', 'relevance_score': '0.4740...",0.474082,-0.082535,1.0,-0.132524,Neutral,Neutral,2
4,Volkswagen and Tesla Square off in Germany in ...,https://www.thestreet.com/investing/volkswagen...,2022-03-07 22:09:00,['Rob Lenihan'],Tesla and Volkswagen will go head-to-head as b...,https://www.thestreet.com/.image/ar_16:9%2Cc_f...,The Street,NaN,www.thestreet.com,"[{'topic': 'Manufacturing', 'relevance_score':...",-0.077281,Neutral,"[{'ticker': 'TSLA', 'relevance_score': '0.5779...",0.577941,-0.077281,1.0,-0.121253,Neutral,Neutral,2


* Rename from summary to text since Bert trainer assumes this by default...

In [ ]:
df.rename({"summary":"text"},axis=1,inplace=True)

### Look at label distribution

In [ ]:
df[['Original_Sentiment_Label_Text','Sentiment_Label_Text']].value_counts(normalize=True)

Original_Sentiment_Label_Text  Sentiment_Label_Text
Neutral                        Neutral                 0.447539
Somewhat_Bullish               Positive                0.252101
Bullish                        Positive                0.121609
Somewhat-Bearish               Negative                0.119208
Bearish                        Negative                0.059544
Name: proportion, dtype: float64

In [ ]:
df['time_published'].min(),df['time_published'].max()

('2022-03-05 15:20:00', '2024-04-08 11:53:16')

## Inference in NLP: Language models predict or analyze text (e.g., answering questions, generate text (i.e ChatGPT), or classify sentiment in our case.
* Tokenizer for NLP: Breaks text into tokens (like words or characters or bytes) for the model.
* Numerical Conversion: Tokens are mapped to numbers, making them understandable to models.

## Approach 1: Manual Approach: Predict sentiment directly using the model and tokenizer without Pipeline
* Load model and tokenizer from model
  * First time you run it it takes a lot longer since it needs to download model
  * to_device is to put it on GPU

In [ ]:
# Model name from Model Hub
model_name = 'yiyanghkust/finbert-tone'
# Load model
model = AutoModelForSequenceClassification.from_pretrained(model_name).to(device)
# Load tokenizer
tokenizer = AutoTokenizer.from_pretrained(model_name)

/usr/local/lib/python3.10/dist-packages/huggingface_hub/utils/_token.py:88: UserWarning: 
The secret `HF_TOKEN` does not exist in your Colab secrets.
To authenticate with the Hugging Face Hub, create a token in your settings tab (https://huggingface.co/settings/tokens), set it as secret in your Google Colab and restart your session.
You will be able to reuse this secret in all of your notebooks.
Please note that authentication is recommended but still optional to access public models or datasets.
  warnings.warn(


config.json:   0%|          | 0.00/533 [00:00<?, ?B/s]

pytorch_model.bin:   0%|          | 0.00/439M [00:00<?, ?B/s]

vocab.txt:   0%|          | 0.00/226k [00:00<?, ?B/s]

##### Note we can look at model config
* label2id gives us label to Id
* id2label gives up id to label

### Attention is all you need. Transformer (AI) revolution!
https://arxiv.org/pdf/1706.03762.pdf

In [ ]:
model.config

BertConfig {
  "_name_or_path": "yiyanghkust/finbert-tone",
  "architectures": [
    "BertForSequenceClassification"
  ],
  "attention_probs_dropout_prob": 0.1,
  "classifier_dropout": null,
  "hidden_act": "gelu",
  "hidden_dropout_prob": 0.1,
  "hidden_size": 768,
  "id2label": {
    "0": "Neutral",
    "1": "Positive",
    "2": "Negative"
  },
  "initializer_range": 0.02,
  "intermediate_size": 3072,
  "label2id": {
    "Negative": 2,
    "Neutral": 0,
    "Positive": 1
  },
  "layer_norm_eps": 1e-12,
  "max_position_embeddings": 512,
  "model_type": "bert",
  "num_attention_heads": 12,
  "num_hidden_layers": 12,
  "pad_token_id": 0,
  "position_embedding_type": "absolute",
  "transformers_version": "4.38.2",
  "type_vocab_size": 2,
  "use_cache": true,
  "vocab_size": 30873
}

In [ ]:
id_2_label = model.config.id2label
id_2_label

{0: 'Neutral', 1: 'Positive', 2: 'Negative'}

* Convert sentence into tokens

In [ ]:
sentence = "The market outlook is very positive thanks to the new economic policies."

inputs = tokenizer(sentence, return_tensors="pt", padding=True, truncation=True, max_length=512)

inputs

{'input_ids': tensor([[   3,    6,   52,  954,   17,  190,  483, 1237,    9,    6,   56,  289,
          693,   48,    4]]), 'token_type_ids': tensor([[0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0]]), 'attention_mask': tensor([[1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1]])}

In [ ]:
# To make all tensors on same device
inputs = {k: v.to(device) for k, v in inputs.items()}

* Use Pytorch or Tensorflow (in this case Pytorch) to make a prediction on the input tokens.


In [ ]:
with torch.no_grad():
    outputs = model(**inputs)
outputs

SequenceClassifierOutput(loss=None, logits=tensor([[-7.4354, 12.1102, -6.3745]], device='cuda:0'), hidden_states=None, attentions=None)

* From the outputs we get a the logit predictions for each class 0,1,2
* Take argmax to find most likely class

In [ ]:
predictions = np.argmax(outputs.logits.cpu().numpy(), axis=1)

predictions

array([1])

### Map this class to its meaning
* 0: Positive
* 1: Negative
* 2: Neutral

In [ ]:
 # Map the array elements using a list comprehension
mapped_array = [id_2_label[element] for element in predictions]

print(mapped_array)


['Positive']


## Approach 2: Transformer Pipeline Approach
* Basically everything in the manual approach for you in 1 step.

* Transformer Pipeline for NLP: Streamlines tasks by:
  - Auto-tokenizing text.
  - Performing model inference (like text analysis or generation).
  - Providing straightforward results, abstracting away technical complexities.

* Don't forget to specify device since we don't want to run on CPU!


#### Load the sentiment analysis pipeline with the FinBERT model
* Note here I specify the model name from model-hub

In [ ]:
# Model name from Model Hub
model_name = 'yiyanghkust/finbert-tone'

sentiment_pipeline = pipeline(task="sentiment-analysis", model=model_name,batch_size=128,device=device)


In [ ]:
sentiment_pipeline(sentence)



[{'label': 'Positive', 'score': 1.0}]

* Can also specify the model and tokenizer to create the pipeline

In [ ]:
sentiment_pipeline = pipeline(task="sentiment-analysis",model=model,tokenizer=tokenizer,batch_size=128,device=device)

sentiment_pipeline(sentence)


[{'label': 'Positive', 'score': 1.0}]

In [ ]:
sentence = "The market outlook is very negative thanks to the new economic policies."
sentiment_pipeline(sentence)

[{'label': 'Negative', 'score': 0.9999997615814209}]

In [ ]:
sentence = "The market outlook is very neutral thanks to the new economic policies."
sentiment_pipeline(sentence)

[{'label': 'Negative', 'score': 0.9921896457672119}]

In [ ]:
sentence = "The market outlook is unknown neutral."
sentiment_pipeline(sentence)

[{'label': 'Positive', 'score': 0.5805525779724121}]

#### Now Let's Make Predictions on Entire DataSet to See How model Performs out of the Box
* Note with CPU only this takes a lot, lot longer!

In [ ]:
preds = sentiment_pipeline(df['text'].tolist())

In [ ]:
preds[0:20]

[{'label': 'Neutral', 'score': 0.9852539896965027},
 {'label': 'Neutral', 'score': 0.9876309037208557},
 {'label': 'Positive', 'score': 0.9768272042274475},
 {'label': 'Negative', 'score': 0.8522449731826782},
 {'label': 'Neutral', 'score': 0.9999992847442627},
 {'label': 'Neutral', 'score': 0.9999985694885254},
 {'label': 'Positive', 'score': 0.9976477026939392},
 {'label': 'Positive', 'score': 0.9963990449905396},
 {'label': 'Neutral', 'score': 0.9168131351470947},
 {'label': 'Negative', 'score': 0.7819403409957886},
 {'label': 'Positive', 'score': 0.9976477026939392},
 {'label': 'Neutral', 'score': 0.9918795228004456},
 {'label': 'Neutral', 'score': 0.9998401403427124},
 {'label': 'Positive', 'score': 0.9938215017318726},
 {'label': 'Positive', 'score': 0.9999901056289673},
 {'label': 'Positive', 'score': 0.7793431282043457},
 {'label': 'Neutral', 'score': 0.9996793270111084},
 {'label': 'Positive', 'score': 0.6943627595901489},
 {'label': 'Positive', 'score': 0.9590197205543518},
 

* Extract just the prediction name from label key

In [ ]:
df['prediction']=[pred['label'] for pred in preds]


* We can see how model is doing


In [ ]:
df.groupby(['Original_Sentiment_Label_Text','prediction']).size()

Original_Sentiment_Label_Text  prediction
Bearish                        Negative       263
                               Neutral        205
                               Positive        28
Bullish                        Negative       116
                               Neutral        522
                               Positive       375
Neutral                        Negative      1018
                               Neutral       2196
                               Positive       514
Somewhat-Bearish               Negative       410
                               Neutral        494
                               Positive        89
Somewhat_Bullish               Negative       352
                               Neutral       1264
                               Positive       484
dtype: int64

In [ ]:
df.groupby(['Sentiment_Label_Text','prediction']).size()

Sentiment_Label_Text  prediction
Negative              Negative       673
                      Neutral        699
                      Positive       117
Neutral               Negative      1018
                      Neutral       2196
                      Positive       514
Positive              Negative       468
                      Neutral       1786
                      Positive       859
dtype: int64

In [ ]:
balanced_accuracy_score(df['Sentiment_Label_Text'],df['prediction'])

0.438992199173226

In [ ]:
accuracy_score(df['Sentiment_Label_Text'],df['prediction'])

0.4475390156062425

##### For fine-tuning we will need indices 0,1,2 for labels instead of names so let's use model.config.label2id to create this colum

In [ ]:
model.config.label2id

{'Positive': 1, 'Negative': 2, 'Neutral': 0}

In [ ]:
df['label']=df['Sentiment_Label_Text'].apply(lambda l:model.config.label2id[l])
df['label'].value_counts()

label
0    3728
1    3113
2    1489
Name: count, dtype: int64

### Split into train/val/test for later comparison.
* For simplicity we split based on time.
  - First 60% train
  - Next 20% val
  - Next 20% test
* This can be problematic a bit since class balance changes over time and some articles on boundries between train/val or val/test have some overlap, but completely beats bias of stratified sample usually used since some articles are literally on same thing, but maybe different sources.


In [ ]:
train_end_point = int(df.shape[0]*0.6)
val_end_point = int(df.shape[0]*0.8)
df_train = df.iloc[:train_end_point,:]
df_val = df.iloc[train_end_point:val_end_point,:]
df_test = df.iloc[val_end_point:,:]
print(df_train.shape, df_test.shape, df_val.shape)

(4998, 22) (1666, 22) (1666, 22)


#### Test set accuracy before fine-tuning

In [ ]:
preds=sentiment_pipeline(df_test['text'].tolist())
df_test['prediction']=[pred['label'] for pred in preds]
balanced_accuracy_score(df_test['Sentiment_Label_Text'],df_test['prediction'])

<ipython-input-41-925fa15f9397>:2: SettingWithCopyWarning: 
A value is trying to be set on a copy of a slice from a DataFrame.
Try using .loc[row_indexer,col_indexer] = value instead

See the caveats in the documentation: https://pandas.pydata.org/pandas-docs/stable/user_guide/indexing.html#returning-a-view-versus-a-copy
  df_test['prediction']=[pred['label'] for pred in preds]


0.4616908988240651

In [ ]:
accuracy_score(df_test['Sentiment_Label_Text'],df_test['prediction'])

0.44717887154861946

### Fine-tuning using trainer class from Hugging Face!
* Convert to huggingface datasets for preparation for fine-tuning

In [ ]:
# Converting pandas DataFrames into Hugging Face Dataset objects:
dataset_train = Dataset.from_pandas(df_train)
dataset_val = Dataset.from_pandas(df_val)
dataset_test = Dataset.from_pandas(df_test)

# Tokenizing the datasets:
dataset_train = dataset_train.map(lambda e: tokenizer(e['text'], truncation=True, padding='max_length', max_length=128), batched=True)
dataset_val = dataset_val.map(lambda e: tokenizer(e['text'], truncation=True, padding='max_length', max_length=128), batched=True)
dataset_test = dataset_test.map(lambda e: tokenizer(e['text'], truncation=True, padding='max_length' , max_length=128), batched=True)

# Setting the dataset format: (needed for Pytorch?)
dataset_train.set_format(type='torch', columns=['input_ids', 'token_type_ids', 'attention_mask', 'label'])
dataset_val.set_format(type='torch', columns=['input_ids', 'token_type_ids', 'attention_mask', 'label'])
dataset_test.set_format(type='torch', columns=['input_ids', 'token_type_ids', 'attention_mask', 'label'])


# Shuffle the training dataset
dataset_train_shuffled = dataset_train.shuffle(seed=42)  # Using a seed for reproducibility

Map:   0%|          | 0/4998 [00:00<?, ? examples/s]

Map:   0%|          | 0/1666 [00:00<?, ? examples/s]

Map:   0%|          | 0/1666 [00:00<?, ? examples/s]

### Define trainer to fine tune model

In [ ]:
def compute_metrics(eval_pred):
    predictions, labels = eval_pred
    predictions = np.argmax(predictions, axis=1)
    return {'balanced_accuracy' : balanced_accuracy_score(predictions, labels),'accuracy':accuracy_score(predictions,labels)}

args = TrainingArguments(
    output_dir='temp/',
    evaluation_strategy='epoch',
    save_strategy='epoch',
    logging_strategy="steps",  # Log every X steps
    logging_steps=50,  # Log every 50 steps
    learning_rate=2e-6,
    per_device_train_batch_size=32,
    per_device_eval_batch_size=32,
    num_train_epochs=3,
    weight_decay=0.1,
    load_best_model_at_end=True,
    metric_for_best_model='balanced_accuracy',
)

trainer = Trainer(
        model=model,                         # the instantiated 🤗 Transformers model to be trained
        args=args,                  # training arguments, defined above
        train_dataset=dataset_train_shuffled,         # training dataset
        eval_dataset=dataset_val,            # evaluation dataset
        compute_metrics=compute_metrics
)





/usr/local/lib/python3.10/dist-packages/accelerate/accelerator.py:436: FutureWarning: Passing the following arguments to `Accelerator` is deprecated and will be removed in version 1.0 of Accelerate: dict_keys(['dispatch_batches', 'split_batches', 'even_batches', 'use_seedable_sampler']). Please pass an `accelerate.DataLoaderConfiguration` instead: 
dataloader_config = DataLoaderConfiguration(dispatch_batches=None, split_batches=False, even_batches=True, use_seedable_sampler=True)
  warnings.warn(


#### Train the model!

In [ ]:
trainer.train()

Epoch,Training Loss,Validation Loss,Balanced Accuracy,Accuracy
1,1.462100,1.275218,0.415967,0.454382
2,1.137800,1.086019,0.433119,0.472389
3,1.063500,1.047432,0.441973,0.487995


TrainOutput(global_step=471, training_loss=1.4677028251048612, metrics={'train_runtime': 361.8291, 'train_samples_per_second': 41.439, 'train_steps_per_second': 1.302, 'total_flos': 986280646353408.0, 'train_loss': 1.4677028251048612, 'epoch': 3.0})

#### Can use trainer.predict method to make predictions on test set.

In [ ]:
predictions = trainer.predict(dataset_test)
predictions

PredictionOutput(predictions=array([[ 0.268502  , -0.27802244,  0.7966725 ],
       [ 0.268502  , -0.27802244,  0.7966725 ],
       [ 0.6718856 , -0.64288   ,  0.17961723],
       ...,
       [ 0.90422726, -0.44965187, -0.79287374],
       [ 0.35220486,  0.61813533,  0.48021153],
       [ 0.74998796,  1.2189636 , -1.6704557 ]], dtype=float32), label_ids=array([0, 0, 0, ..., 2, 0, 0]), metrics={'test_loss': 1.0325454473495483, 'test_balanced_accuracy': 0.44133516154107433, 'test_accuracy': 0.4645858343337335, 'test_runtime': 11.5845, 'test_samples_per_second': 143.813, 'test_steps_per_second': 4.575})

##### Can also use trained model as part of a pipeline by
* Saving locally
* Uploading to Model Hub

Here we save locally

In [ ]:
model_path = "path/to/save/model"


# Save the model
trainer.model.save_pretrained(model_path)

# Save the tokenizer associated with the model
# Save the tokenizer
tokenizer.save_pretrained(model_path)



('path/to/save/model/tokenizer_config.json',
 'path/to/save/model/special_tokens_map.json',
 'path/to/save/model/vocab.txt',
 'path/to/save/model/added_tokens.json',
 'path/to/save/model/tokenizer.json')

###### Load the trained model into a pipeline.  

In [ ]:
trained_pipeline = pipeline("text-classification", model=model_path, tokenizer=model_path,device=device)

##### Make Predictions and evaluate accuracy

In [ ]:
preds=trained_pipeline(df_test['text'].tolist())
df_test['prediction']=[pred['label'] for pred in preds]

<ipython-input-49-cad1efb34115>:2: SettingWithCopyWarning: 
A value is trying to be set on a copy of a slice from a DataFrame.
Try using .loc[row_indexer,col_indexer] = value instead

See the caveats in the documentation: https://pandas.pydata.org/pandas-docs/stable/user_guide/indexing.html#returning-a-view-versus-a-copy
  df_test['prediction']=[pred['label'] for pred in preds]


##### Calculate the balanced accuracy score

In [ ]:
# Calculate the balanced accuracy score
score = balanced_accuracy_score(df_test['Sentiment_Label_Text'], df_test['prediction'])
print(f"Balanced Accuracy Score: {score}")


Balanced Accuracy Score: 0.4338598903499246


In [ ]:
# Calculate the balanced accuracy score
score = accuracy_score(df_test['Sentiment_Label_Text'], df_test['prediction'])
print(f"Accuracy Score: {score}")


Balanced Accuracy Score: 0.4645858343337335
